In [ ]:
# IMPORTS
from egomimic.rldb.utils import *
import torch
import numpy as np
from egomimic.utils.egomimicUtils import CameraTransforms, draw_actions
import torchvision.io as io
import os

In [ ]:
# Load dataset
root = "/coc/flash7/paphiwetsa3/datasets/eva_test_data2/proc2/lerobot_test"
repo_id = "rpuns/aria_laundry_rl2"

episodes = [0, 1]
dataset = RLDBDataset(
    repo_id=repo_id, root=root, local_files_only=True, episodes=episodes, mode="sample"
)

In [ ]:
# Get metadata
print(dataset.meta.info["features"])

image_key = "observations.images.front_img_1"
actions_key = "actions_cartesian"

In [ ]:
print(dataset.embodiment)

In [ ]:
# Make data_loader
data_loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

In [19]:
camera_transforms = CameraTransforms(intrinsics_key="base", extrinsics_key="x5Nov18_3")

In [ ]:
camera_transforms.extrinsics["left"]

In [21]:
def visualize_actions(ims, actions, extrinsics, intrinsics, arm="both"):
    for b in range(ims.shape[0]):
        if actions.shape[-1] == 7 or actions.shape[-1] == 14:
            ac_type = "joints"
        elif actions.shape[-1] == 3 or actions.shape[-1] == 6:
            ac_type = "xyz"
        else:
            raise ValueError(f"Unknown action type with shape {actions.shape}")

        ims[b] = draw_actions(ims[b], "xyz", "Purples", actions[b], extrinsics, intrinsics, arm=arm)

    return ims

In [ ]:
import os
import torch
import numpy as np
import cv2

save_dir = "./visualization/"
os.makedirs(save_dir, exist_ok=True)

num_batches = 1
fps = 30

for i, data in enumerate(data_loader):
    if i > num_batches:
        break

    ims = data[image_key].cpu().numpy()
    actions = data[actions_key].cpu().numpy()
    print(actions)

    for b in range(ims.shape[0]):
        video = ims[b]
        video = np.transpose(video, (3, 1, 2, 0))
        video = (video * 255.0).astype(np.uint8)

        height, width = video[0].shape[:2]
        save_path = os.path.join(save_dir, f"batch_{i}_sample_{b}.mp4")

    ims_viz = visualize_actions(ims, actions[:, :3], camera_transforms.extrinsics, camera_transforms.intrinsics)

        for frame in video:
            frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            out.write(frame_bgr)

        out.release()
        print(f"Saved video: {save_path}")
